# JayBee Lookbook generator (CatVTON, free & open-source)
Runs on a **free Google Colab GPU** (Runtime > Change runtime type > T4 GPU).
Photos stay inside your own Colab session. Nothing is sent to any paid service.

**Before you start**
1. Prepare 3-10 *model* photos (full or upper body, person facing the camera, plain light background works best).
2. Prepare your garment images: front view on a plain/white background. Name them with the garment type at the end:
   `jaybee_tee__upper.png`, `jaybee_hoodie__upper.png`, `jaybee_pants__lower.png`, `jaybee_dress__overall.png`

> This notebook was written without being able to run it. If a cell fails, send the error text back and it will be fixed.

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/Zheng-Chong/CatVTON /content/CatVTON
%cd /content/CatVTON
!pip install -q -r requirements.txt

## 1. Upload model photos

In [ ]:
import os, shutil
from google.colab import files
os.makedirs('/content/models', exist_ok=True)
up = files.upload()
for n in up: shutil.move(n, '/content/models/' + n)
print(os.listdir('/content/models'))

## 2. Upload garment images (names end with __upper / __lower / __overall)

In [ ]:
os.makedirs('/content/garments', exist_ok=True)
up = files.upload()
for n in up: shutil.move(n, '/content/garments/' + n)
print(os.listdir('/content/garments'))

## 3. Load the model (downloads weights once, about 10 GB)

In [ ]:
import torch, numpy as np
from PIL import Image
from huggingface_hub import snapshot_download
from diffusers.image_processor import VaeImageProcessor
from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

repo_path = snapshot_download(repo_id="zhengchong/CatVTON")
pipeline = CatVTONPipeline(
    base_ckpt="booksforcharlie/stable-diffusion-inpainting",
    attn_ckpt=repo_path, attn_ckpt_version="mix",
    weight_dtype=init_weight_dtype("fp16"),   # T4 has no bf16
    use_tf32=False, device="cuda")
mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)
automasker = AutoMasker(densepose_ckpt=os.path.join(repo_path, "DensePose"),
                        schp_ckpt=os.path.join(repo_path, "SCHP"), device="cuda")
print("ready")

## 4. Generate every model x garment combination

In [ ]:
import json, glob
os.makedirs('/content/out', exist_ok=True)
manifest = []
STEPS, GUIDE, SEED = 40, 2.5, 42
for mp in sorted(glob.glob('/content/models/*')):
    mname = os.path.splitext(os.path.basename(mp))[0]
    person = Image.open(mp).convert("RGB")
    person_r = resize_and_crop(person, (768, 1024))
    person_r.save(f'/content/out/{mname}.jpg', quality=92)
    for gp in sorted(glob.glob('/content/garments/*')):
        gname = os.path.splitext(os.path.basename(gp))[0]
        ctype = gname.rsplit('__', 1)[-1] if '__' in gname else 'upper'
        title = gname.rsplit('__', 1)[0].replace('_', ' ').title()
        cloth = resize_and_padding(Image.open(gp).convert("RGB"), (768, 1024))
        mask = automasker(person_r, ctype)['mask']
        mask = mask_processor.blur(mask, blur_factor=9)
        gen = torch.Generator(device="cuda").manual_seed(SEED)
        res = pipeline(image=person_r, condition_image=cloth, mask=mask,
                       num_inference_steps=STEPS, guidance_scale=GUIDE, generator=gen)[0]
        out = f'/content/out/{mname}__{gname}.jpg'
        res.save(out, quality=92)
        manifest.append({"title": title, "before": f"lookbook/{mname}.jpg", "after": f"lookbook/{mname}__{gname}.jpg"})
        print("done", out)
json.dump(manifest, open('/content/out/manifest.json', 'w'), ensure_ascii=False, indent=1)

## 5. Download results
Unzip and upload everything into the `lookbook/` folder of your website repository (replace `manifest.json`). The Lookbook section appears automatically.

In [ ]:
!cd /content/out && zip -qr /content/lookbook.zip .
files.download('/content/lookbook.zip')